In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import OrderedDict
import pandas as pd

from src.checkpoint import RESULT_DIR, MODEL_DIR, STAT_DIR
from src.task_selectors.factory import get_selector_name

stat_directory = RESULT_DIR + STAT_DIR

In [ ]:
class Run():
    def __init__(self, filename: str):
        filename_splits = filename.split("_")
        self.env_name = filename_splits[0]
        self.alg_name = filename_splits[1]
        self.seed = int(filename_splits[2])
        self.selector = get_selector_name(int(filename_splits[3]))
        self.id = filename_splits[4].split(".")[0]
        self.filename = filename
        
        # load and sort stats
        with open(f"{stat_directory}{filename}", "r") as f:
            self.stats: dict = json.load(f)
            self.stats_loc = self.stats.pop("loc")
        self.stats_generic: dict[str, list[float]] = {}
        self.stats_envs: dict[str, dict[str, list[float]]] = {}
        self.stats_envs_loc: dict[str, dict[str, list[int]]] = {}
        self.adapted_reward: pd.DataFrame = pd.DataFrame(data=[], index=np.arange(6000)[1:])
        for k, v in self.stats.items():
            splits = str(k).split(" ")
            if splits[0] == "Env":
                stat = " ".join(splits[2:])
                env_index = int(splits[1])
                if stat not in self.stats_envs.keys():
                    self.stats_envs[stat] = {}
                    self.stats_envs_loc[stat] = {}
                self.stats_envs[stat][env_index] = v
                self.stats_envs_loc[stat][env_index] = self.stats_loc[k]
                if stat == "adapted_reward":
                    print(min(self.stats_loc[k]), max(self.stats_loc[k]))
                    self.adapted_reward.insert(min(len(self.adapted_reward.columns), env_index), env_index, pd.Series(v, index=self.stats_loc[k]))
            else:
                self.stats_generic[k] = v
        np.random.seed(self.seed)
        self.velocities = np.random.uniform(0.0, 10.0, 20).tolist()
        self.adapted_reward = self.adapted_reward.reindex(sorted(self.adapted_reward.columns), axis=1)
        self.adapted_reward.columns = self.velocities
        self.adapted_reward = self.adapted_reward.reindex(sorted(self.adapted_reward.columns), axis=1)
        self.selections = self.adapted_reward.count()
        self.adapted_reward = self.adapted_reward.interpolate('index')
        self.adapted_reward['avg'] = self.adapted_reward.mean(axis=1)
        for k, v in self.stats_envs.items():
            self.stats_envs[k] = OrderedDict(sorted(v.items()))
        
    def __str__(self):
        return f"{self.selector} with seed {self.seed} - {self.id}"
            

In [ ]:
run_ids = [f for f in os.listdir(stat_directory) if f.split("_")[0] != "Eval"]
runs = [Run(name) for name in run_ids]
for r in runs:
    pass

In [ ]:
def plot(df):
    plt.show()
    plt.figure(figsize=(15,8))
    plt.plot(df)
    plt.title("adapted_reward")
    plt.legend()
    plt.grid()
    plt.show()

def plot_average_reward(runs: list[Run]):
    reward_dfs = [run.adapted_reward for run in runs]
    df = pd.concat(reward_dfs).groupby(level=0).mean()
    for r in reward_dfs:
        plot(r)
    plot(df)

In [ ]:
# Bin heights
counts = [5, 8, 3]

# Desired x positions
x_positions = [0.3, 1.7, 2.3]

# Bar widths
widths = [0.2, 0.2, 0.2]

plt.bar(x_positions, counts, width=widths)

plt.xlabel("Velocity")
plt.ylabel("Count")

plt.show()

In [ ]:
for run in runs:
    r = run.selections.copy()
    
    """r.index = np.arange(20)
    selection = []
    for k, v in r.items():
        selection += [k for i in range(v)]
    counts, bins, _ = plt.hist(selection, bins=20, align='left')
    plt.xticks(np.arange(20) * 0.95, [f"{round(v, 2)}" for v in sorted(run.velocities)], rotation=45)"""
    
    plt.bar(np.arange(20), r.values, width=1)
    
    title = f"Selected Environments - {run.selector}"
    plt.title(title)
    plt.ylabel("Number of selections")
    plt.xlabel("Environment velocity (units/s)")
    #plt.savefig(f"plots/{title} - Seed {run.seed}", bbox_inches='tight')
    plt.show()

In [ ]:
def plot_bin_probs(runs: list[Run]):
    for r in runs:
        selection = []  
        print(r)
        for k, v in r.selections.items():
            selection += [k for i in range(v)]
        plt.hist(selection, bins=10)
        title = f"Selected environment velocities - {r.selector}"
        plt.title(title)
        plt.xlabel("Velocity")
        #plt.savefig(f"plots/{title}")
        plt.show()

for i in range(2):
    plot_bin_probs([run for run in runs if run.selector == get_selector_name(i)])

In [ ]:
runs = sorted(runs, key=lambda x: f"{x.selector}{x.seed}", reverse=True)
plot_average_reward([run for run in runs if run.selector == get_selector_name(1)])

In [ ]:
def plot_stat(run: Run, values: dict, title, loc=None):
    def plot_line(values, label, loc=None, color=None):
        if loc is None:
            plt.plot(values, label=label, color=color)
        else:
            plt.plot(loc, values, label=label, color=color)
            
    #plt.figure(figsize=(15,8))
    for k, v in values.items():
        color = plt.cm.tab20(k / 20)
        plot_line(v, round(run.velocities[k], 2), loc[k], color)
    plt.title(title)
    plt.xlabel("Episode")
    plt.ylabel("Reward")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid()
    plt.savefig(f"plots/{title} - Seed {run.seed}", bbox_inches='tight')
    plt.show()

def plot_envs(run: Run, envs: list[int] = None, flattenings: int = 0, l: int = 60000):
    to_plot = {}
    to_plot_loc = {}
    for vel in sorted(run.velocities):
        # find index of velocity
        env = [k for k, v in enumerate(run.velocities) if v == vel][0]
        v = run.stats_envs["adapted_reward"][env]
        if envs is None or env in envs:
            for _ in range(flattenings):
                v =  v[0:1] + [(v[i-1] + v[i] + v[i+1]) / 3 for i in range(1, len(v) - 1)] + v[-2:-1]
            early_mask = np.array(run.stats_envs_loc["adapted_reward"][env]) < l
            to_plot[env] = np.array(v)[early_mask]
            to_plot_loc[env] = np.array(run.stats_envs_loc["adapted_reward"][env])[early_mask]
    plot_stat(run, to_plot, f"Reward after Training Adaptation - {run.selector}", to_plot_loc)

for r in runs:
    if r.id != "1773303826" and r.id != "1773303858":
        pass
    print(r)
    plot_envs(r, flattenings=100)